# Lab 06 — Clean & Standardize · Join & Aggregate
### Week 2 · Data Engineering for LLM Pipelines

This is an **integration lab**. It revisits the moves from Labs 04–05 — cleaning,
joining, aggregating — and wires them into one reproducible pipeline that turns a
**messy** customer profile table plus an orders fact table into tidy, LLM-ready
feature rollups.

The data (synthetic, clearly-fictional **Cordwell Home & Hardware**) is deliberately
dirty: inconsistent country spellings, currency written six different ways, dates in
four formats, and missing values. Your job is to make it trustworthy, then join and
summarize it — *without* silently corrupting the numbers.

**By the end you will be able to:**
1. Build a reproducible cleaning pipeline: normalize categories against a **reference
   dimension**, parse messy **currency** to numeric, and parse mixed-format **dates**
   safely.
2. Engineer **vectorized** features (`is_adult`, `is_high_value`) — no `apply`.
3. **Inner-join** orders to customers with a cardinality guard, and triage unmatched
   rows with an **anti-join**.
4. Produce **per-segment** and **per-customer** rollups and persist them to Parquet.

> **Heads-up — the hints are lighter now.** Each Part opens with a **Toolbox** of
> methods you'll need *somewhere* in that Part; figuring out which tool goes where is
> part of the exercise. The `check()` cells still score you (target **28/28**), and a
> red check never halts the notebook — it just marks what's left. Some traps here are
> *silent*: code that runs fine and still gives a wrong answer. Watch the counts.


## Setup — build the messy dataset & the `check()` helper

In [ ]:
%pip install -r requirements.txt

In [ ]:
import warnings
import numpy as np
import pandas as pd

print("pandas", pd.__version__, "| numpy", np.__version__)

def build_cordwell_raw(seed=2025):
    """Synthetic, clearly-fictional Cordwell customer profiles + orders — deliberately messy."""
    rng = np.random.default_rng(seed)
    n = 1000

    country_choices = ['US','U.S.A.','usa','United States','SG','sg','DE','Germany',
                       'Brasil','BR','Deutschland','Brazil','N/A']
    country_p = np.array([.24,.05,.05,.08,.10,.03,.10,.05,.06,.05,.03,.04,.05]); country_p/=country_p.sum()

    base = np.round(rng.lognormal(6.0, 1.0, size=n), 2)          # true lifetime value (USD)
    def encode(v, style):
        if style == 0: return f"${v:,.2f}"                                    # $1,234.50
        if style == 1:
            s = f"{v:,.2f}"; return "€" + s.replace(",", "§").replace(".", ",").replace("§", ".")  # €1.234,50
        if style == 2:
            s = f"{v:,.2f}"; return "R$ " + s.replace(",", "§").replace(".", ",").replace("§", ".")     # R$ 1.234,50
        if style == 3: return f"USD {v:.2f}"                                  # USD 99.95
        if style == 4: return f"{v:,.0f}"                                     # 1,234  (cents dropped)
        return f"{v:.2f}"
    styles = rng.choice(5, size=n, p=[.34,.14,.10,.16,.26])
    ltv_raw = np.array([encode(v, st) for v, st in zip(base, styles)], dtype=object)
    r = rng.random(n)
    ltv_raw[r < 0.04] = ''
    ltv_raw[r > 0.98] = None

    date_choices = np.array(['2025-01-05','01/06/2025','2025/01/07','06-01-2025', None], dtype=object)
    date_p       = np.array([.30,.25,.20,.20,.05])

    customers_raw = pd.DataFrame({
        "customer_id": [f"C{i:05d}" for i in range(1, n+1)],
        "email": [f"acct{i}@cordwell-accounts.example" if rng.random() > 0.02 else None for i in range(1, n+1)],
        "age": rng.integers(16, 80, size=n).astype("float64"),
        "country": rng.choice(country_choices, size=n, p=country_p),
        "signup_date": rng.choice(date_choices, size=n, p=date_p),
        "lifetime_value": ltv_raw,
    })

    m = 3000
    orders = pd.DataFrame({
        "order_id": np.arange(50001, 50001+m),
        "customer_id": rng.choice(customers_raw["customer_id"], size=m),
        "order_date": rng.choice(['2025-01-06','2025-01-07','2025-01-08','2025-01-09'], size=m),
        "freight": np.round(rng.lognormal(3.3, 0.6, size=m), 2),
    })

    country_dim = pd.DataFrame({
        "raw": ['USA','U.S.A.','United States','US','usa','U. S. A.','Brasil','BR',
                'Germany','DE','sg','Singapore','SG','N/A'],
        "canonical": ['USA','USA','USA','USA','USA','USA','BR','BR','DE','DE','SG','SG','SG','UNKNOWN'],
    })
    return customers_raw, orders, country_dim

customers_raw, orders, country_dim = build_cordwell_raw()
print("customers_raw:", customers_raw.shape, "| orders:", orders.shape)
display(customers_raw.head())
display(country_dim)

In [ ]:
# ── soft self-check: prints PASS/FAIL, never raises ──────────────────────────
_score = {"pass": 0, "fail": 0}
def check(label, predicate):
    try:
        ok = bool(predicate() if callable(predicate) else predicate); note = ""
    except Exception as e:
        ok, note = False, f"  [error: {type(e).__name__}: {e}]"
    _score["pass" if ok else "fail"] += 1
    print(f"{'✅ PASS' if ok else '❌ FAIL'} — {label}{note}")
def score():
    t = _score["pass"] + _score["fail"]
    print(f"\n{'='*46}\n  {_score['pass']}/{t} checks passing  ({_score['fail']} to go)\n{'='*46}")
def approx(a, b, tol=0.01):
    return abs(float(a) - float(b)) <= tol

---
## Part A — Clean & Standardize

Goal: turn `customers_raw` into a clean `users2` frame with trustworthy
`country_norm`, `ltv_usd`, `signup_dt`, `is_adult`, and `is_high_value` columns.

> **🧰 Toolbox for Part A** — you'll need some of these:
> `.dropna(subset=...)` · `.astype("string")` · `.str.replace(...)` ·
> `.str.upper()` · `.str.fullmatch(...)` · `.merge(..., how="left")` ·
> `.fillna(...)` · `.groupby(...).transform("median")` · `pd.to_numeric(...,
> errors="coerce")` · `pd.to_datetime(..., errors="coerce", format=...)` ·
> `Series.quantile(...)` · vectorized comparisons (`>=`).
>
> ⚠️ Two of these steps have a **silent** failure mode in pandas 3.0 — they run
> without error and still return wrong data. The checks will catch them; your job is
> to work out *why*.

### A1 — Required-field filter

Some downstream steps assume every row has a **customer_id** and an **email**. Drop
rows missing either, into `users1`.

In [ ]:
required = ["customer_id", "email"]
users1 = customers_raw.dropna(subset=required).copy()
print(f"{len(customers_raw)} -> {len(users1)} rows  ({len(customers_raw)-len(users1)} dropped)")

In [ ]:
check("A1: dropped the 22 rows missing email", lambda: len(users1) == 978)
check("A1: customer_id + email fully populated",
      lambda: users1[["customer_id","email"]].notna().all().all())

### A2 — Normalize `country` against the reference dimension

Raw country values are written many ways (`US`, `U.S.A.`, `usa`, `United States`…).
Map each to a canonical code using **`country_dim`** — a real *dimension lookup*, not a
hard-coded dict. Values that don't match anything become **`"UNKNOWN"`**. Add the
result as **`country_norm`** on `users1`.

*Think:* the raw spellings won't equal the dimension's spellings exactly. What has to
happen to **both** sides before a join key will match?

In [ ]:
def norm_key(s):
    """Fold a country string to a match key: drop '.' and spaces, uppercase."""
    return (s.astype("string")
             .str.replace(".", "", regex=False)
             .str.replace(" ", "", regex=False)
             .str.upper())

ref = (country_dim.assign(raw_key=norm_key(country_dim["raw"]))[["raw_key","canonical"]]
       .drop_duplicates("raw_key"))
users1["country_key"] = norm_key(users1["country"])
users1 = users1.merge(ref, left_on="country_key", right_on="raw_key", how="left")
users1["country_norm"] = users1["canonical"].fillna("UNKNOWN")
users1["country_norm"].value_counts()

In [ ]:
check("A2: US / U.S.A. / usa all collapse to USA", lambda: users1["country_norm"].value_counts().get("USA", 0) == 421)
check("A2: unmatched spellings became UNKNOWN (146)", lambda: users1["country_norm"].value_counts().get("UNKNOWN", 0) == 146)
check("A2: exactly 5 canonical codes present",
      lambda: set(users1["country_norm"].unique()) == {"USA","DE","SG","BR","UNKNOWN"})

### A3 — Parse messy currency → numeric `ltv_usd`

`lifetime_value` arrives as strings in mixed conventions. Implement `parse_money` to
the contract in its docstring, then **impute** the missing values: fill with the
**median within `country_norm`**, and anything still missing with `0.0`.

*The hard part is the decimal separator:* `"45,00"` means 45 (comma-decimal), but
`"1,234"` means 1234 (comma-thousands). Same character, opposite meaning.

In [ ]:
def parse_money(series: pd.Series) -> pd.Series:
    """Parse messy currency strings to float USD. Rules:
        "$1,234.50" -> 1234.50     (US: comma=thousands, dot=decimal)
        "€1.234,50" -> 1234.50     (EU: dot=thousands, comma=decimal)
        "R$ 45,00"  -> 45.00       (comma=decimal)
        "USD 99.95" -> 99.95
        "1,234"     -> 1234.0      (comma=thousands, no decimal)
        "" / None   -> NaN
    """
    s = series.astype("string").str.strip()
    s = (s.str.replace("USD","",regex=False).str.replace("EUR","",regex=False)
          .str.replace("R$","",regex=False).str.replace("$","",regex=False)
          .str.replace("€","",regex=False).str.replace("£","",regex=False)
          .str.replace(" ","",regex=False))
    m_dec  = s.str.fullmatch(r"\d+,\d{1,2}").fillna(False)                  # 45,00  -> comma is decimal
    m_both = s.str.fullmatch(r"\d{1,3}(\.\d{3})+,\d{1,2}").fillna(False)    # 1.234,50 -> EU thousands+decimal
    s = s.where(~m_dec,  s.str.replace(",", ".", regex=False))
    s = s.where(~m_both, s.str.replace(".", "", regex=False).str.replace(",", ".", regex=False))
    s = s.str.replace(",", "", regex=False)                                 # leftover commas = thousands
    return pd.to_numeric(s, errors="coerce")

users1["ltv_usd"] = parse_money(users1["lifetime_value"])
med = users1.groupby("country_norm")["ltv_usd"].transform("median")
users1["ltv_usd"] = users1["ltv_usd"].fillna(med).fillna(0.0)
users1["ltv_usd"].describe()

In [ ]:
check("A3: US format \"$1,234.50\" -> 1234.50",
      lambda: approx(parse_money(pd.Series(["$1,234.50"]))[0], 1234.50))
check("A3: EU format \"€1.234,50\" -> 1234.50",
      lambda: approx(parse_money(pd.Series(["€1.234,50"]))[0], 1234.50))
check("A3: comma-decimal \"R$ 45,00\" -> 45.00 (not 4500)",
      lambda: approx(parse_money(pd.Series(["R$ 45,00"]))[0], 45.00))
check("A3: no missing ltv_usd after imputation", lambda: int(users1["ltv_usd"].isna().sum()) == 0)

### A4 — Parse dates safely, then engineer features

Parse `signup_date` into a real datetime `signup_dt`, **drop** rows with no usable
date into `users2`, then add two **vectorized** boolean features:
`is_adult` (age ≥ 18) and `is_high_value` (`ltv_usd` in the **top decile** — at or
above the 90th percentile).

> ⚠️ **This is the silent one.** `signup_date` holds four different formats. The
> obvious one-liner parses **cleanly, with no error**, and quietly turns most of your
> dates into `NaT`. If `users2` comes out far smaller than ~920 rows, that's the trap
> — find the parse that keeps every real date.

In [ ]:
# The trap, made visible: infer-from-first-row vs. per-element parsing
naive = pd.to_datetime(users1["signup_date"], errors="coerce")
mixed = pd.to_datetime(users1["signup_date"], errors="coerce", format="mixed")
print(f"naive parse -> {naive.isna().sum()} NaT   |   format='mixed' -> {mixed.isna().sum()} NaT")

users1["signup_dt"] = mixed
users2 = users1.dropna(subset=["signup_dt"]).copy()

users2["is_adult"] = users2["age"] >= 18
q90 = users2["ltv_usd"].quantile(0.90)
users2["is_high_value"] = users2["ltv_usd"] >= q90
print(f"users2: {len(users2)} rows | 90th-pct LTV = ${q90:,.2f} | "
      f"adults={users2['is_adult'].sum()} | high-value={users2['is_high_value'].sum()}")

In [ ]:
check("A4: kept every real date (~923 rows, not ~200)", lambda: len(users2) == 923)
check("A4: 90th-percentile LTV threshold ≈ 1343.13", lambda: approx(users2["ltv_usd"].quantile(0.90), 1343.13, tol=0.5))
check("A4: is_adult is a real boolean feature (895 adults)",
      lambda: users2["is_adult"].dtype == bool and int(users2["is_adult"].sum()) == 895)
check("A4: is_high_value flags the top decile (~10%)",
      lambda: 9.0 <= 100*users2["is_high_value"].mean() <= 11.0 and int(users2["is_high_value"].sum()) == 93)

### A5 — Export the cleaned table

Write `users2` to `artifacts/clean/users_clean.parquet` (no index).

In [ ]:
from pathlib import Path
out = Path("artifacts/clean"); out.mkdir(parents=True, exist_ok=True)
users2.to_parquet(out / "users_clean.parquet", index=False)
print("wrote", out / "users_clean.parquet", "|", users2.shape)

In [ ]:
check("A5: users_clean.parquet exists", lambda: (Path("artifacts/clean")/"users_clean.parquet").exists())

---
## Part B — Join & Aggregate

Goal: join `orders` to the cleaned customers, triage what doesn't match, and roll up
per-segment and per-customer metrics.

> **🧰 Toolbox for Part B** — `.drop_duplicates(subset=...)` ·
> `.merge(..., how="inner"|"left", validate=..., indicator=True)` ·
> `np.select(conditions, choices, default=...)` · `.groupby(...).agg(named=...)` ·
> `("col", "nunique")` · `Series.mean()` on a boolean mask.
>
> Remember from Lab 05: pick the join by what you do with non-matches, and a
> cardinality **guard** turns a silent fan-out into a loud error.

### B1 — Build the customer dimension & inner-join

Project the columns you need from `users2`, ensure **one row per customer**, then
inner-join `orders` to it — and **assert the cardinality you expect** so a duplicate
customer can never fan-out your freight totals.

In [ ]:
customers = (users2[["customer_id","email","country_norm","signup_dt",
                     "is_adult","is_high_value","ltv_usd"]]
             .drop_duplicates(subset=["customer_id"]))

joined = orders.merge(customers, on="customer_id", how="inner", validate="many_to_one")
print(f"orders={len(orders)} | customers={len(customers)} | joined={len(joined)}")
joined.head()

In [ ]:
check("B1: one row per customer in the dimension", lambda: customers["customer_id"].is_unique)
check("B1: inner join keeps only matched orders (2765)", lambda: len(joined) == 2765)

### B2 — Anti-join: which orders lost their customer?

Cleaning **filtered out** some customers (bad email, unparseable date), which orphans
every order that referenced them. Compute the **anti-join rate**: the fraction of
`orders` with no matching row in `customers`. Put the fraction in `anti_rate`.

In [ ]:
tagged = orders.merge(customers[["customer_id"]], on="customer_id", how="left", indicator=True)
anti = tagged[tagged["_merge"] == "left_only"]
anti_rate = (tagged["_merge"] == "left_only").mean()
print(f"orphaned orders: {len(anti)}  |  anti-join rate: {anti_rate:.2%}")

In [ ]:
check("B2: 235 orphaned orders identified",
      lambda: int((orders.merge(customers[["customer_id"]], on="customer_id", how="left", indicator=True)["_merge"]=="left_only").sum()) == 235)
check("B2: anti-join rate ≈ 7.8%", lambda: approx(anti_rate, 0.0783, tol=0.001))

### B3 — Per-segment rollup

Assign each order a **`segment`** from the customer flags — **`high_value`** takes
precedence over **`adult`**, everything else is **`general`** — then aggregate per
**(`country_norm`, `segment`)**: order count, mean & total freight, and **distinct
customers**.

*Think:* precedence between two boolean conditions with a default is exactly what one
NumPy function expresses in a single call.

In [ ]:
seg = joined.assign(segment=np.select(
    [joined["is_high_value"], joined["is_adult"]],
    ["high_value", "adult"],
    default="general"))

per_segment = (seg.groupby(["country_norm","segment"], as_index=False)
    .agg(orders=("order_id","count"),
         freight_mean=("freight","mean"),
         freight_sum=("freight","sum"),
         customers=("customer_id","nunique"))
    .sort_values(["country_norm","orders"], ascending=[True, False], ignore_index=True))
per_segment

In [ ]:
check("B3: segment labels are exactly {high_value, adult, general}",
      lambda: set(seg["segment"].unique()) == {"high_value","adult","general"})
check("B3: one row per (country_norm, segment) — 15 rows", lambda: len(per_segment) == 15)
check("B3: order counts reconcile to the join (2765)", lambda: int(per_segment["orders"].sum()) == 2765)

### B4 — Per-customer rollup, enriched

Aggregate `joined` to **one row per customer** (`n_orders`, mean & total freight),
then join the customer attributes back on. Guard it **one-to-one**.

In [ ]:
per_cust = (joined.groupby("customer_id", as_index=False)
    .agg(n_orders=("order_id","count"),
         freight_mean=("freight","mean"),
         freight_sum=("freight","sum")))

per_cust_enriched = per_cust.merge(customers, on="customer_id", how="left", validate="one_to_one")
print("per-customer rows:", len(per_cust_enriched))
per_cust_enriched.head()

In [ ]:
check("B4: one row per matched customer (873)", lambda: len(per_cust_enriched) == 873)
check("B4: n_orders reconciles to the join (2765)", lambda: int(per_cust_enriched["n_orders"].sum()) == 2765)

### B5 — Persist the rollups

Write `per_segment` and `per_cust_enriched` to Parquet under `artifacts/clean/`.

In [ ]:
per_segment.to_parquet(out / "per_segment.parquet", index=False)
per_cust_enriched.to_parquet(out / "per_customer_enriched.parquet", index=False)
print("wrote per_segment.parquet and per_customer_enriched.parquet")

In [ ]:
check("B5: both rollup parquets written",
      lambda: (Path("artifacts/clean")/"per_segment.parquet").exists()
          and (Path("artifacts/clean")/"per_customer_enriched.parquet").exists())

---
## Stretch goals (optional)

Fast finishers only. Lighter hints still.

> **🧰 Toolbox** — `pd.qcut(..., duplicates="drop")` · `assert` · f-strings ·
> `df.to_parquet(dir, partition_cols=[...])` · `Series.astype(str)`.

### S1 — LTV decile bands

Replace the single high-value cutoff with **deciles**. Cut `ltv_usd` (clipped at 0)
into 10 quantile bands as `ltv_decile` on `users2`. (What happens to the bands if a
lot of customers share the same value? Handle it.)

In [ ]:
users2 = users2.assign(
    ltv_decile=pd.qcut(users2["ltv_usd"].clip(lower=0), q=10, labels=False, duplicates="drop"))
users2["ltv_decile"].value_counts().sort_index()

In [ ]:
check("S1: LTV split into 10 decile bands", lambda: users2["ltv_decile"].nunique() == 10)

### S2 — Fail-fast validation

Real pipelines assert their invariants. Write **assertions** (they must *pass* on your
clean data) that: (a) the anti-join rate is under 10%, and (b) no `n_orders` in
`per_cust_enriched` is null. Set `validation_ok = True` only if both hold.

In [ ]:
validation_ok = False
assert anti_rate < 0.10, f"anti-join rate too high: {anti_rate:.2%}"
assert per_cust_enriched["n_orders"].notna().all(), "n_orders contains nulls"
validation_ok = True
print("all data-quality assertions passed")

In [ ]:
check("S2: fail-fast validations pass on clean data", lambda: validation_ok is True)

### S3 — Partitioned output

Write `per_segment` partitioned by `country_norm` (directory-style) under
`artifacts/clean/per_segment_parts/`, then confirm you wrote one partition per
distinct country.

In [ ]:
import shutil
parts_dir = out / "per_segment_parts"
if parts_dir.exists(): shutil.rmtree(parts_dir)
per_segment.to_parquet(parts_dir, partition_cols=["country_norm"])
n_parts = len(sorted(parts_dir.glob("country_norm=*")))
print("partitions written:", n_parts)

In [ ]:
check("S3: one partition per country_norm (5)", lambda: n_parts == per_segment["country_norm"].nunique() == 5)

### S4 — LLM-ready summary column

Compose a compact natural-language `summary` string per customer for prompt
conditioning — e.g. *"Customer C00007 — high_value buyer in USA — 4 orders, $312.40
total freight."* Build it on `per_cust_enriched` into `llm_view` with columns
`customer_id`, `country_norm`, `summary`.

In [ ]:
seg_label = np.select([per_cust_enriched["is_high_value"], per_cust_enriched["is_adult"]],
                      ["high_value","adult"], default="general")
llm_view = per_cust_enriched.assign(segment=seg_label).assign(
    summary=lambda d: ("Customer " + d["customer_id"]
                       + " — " + d["segment"] + " buyer in " + d["country_norm"]
                       + " — " + d["n_orders"].astype(str) + " orders, $"
                       + d["freight_sum"].round(2).astype(str) + " total freight.")
)[["customer_id","country_norm","summary"]]
print(llm_view["summary"].iloc[0])
llm_view.head()

In [ ]:
check("S4: a summary sentence was built for every customer",
      lambda: llm_view["summary"].str.len().gt(0).all() and "buyer in" in llm_view["summary"].iloc[0])
score()

---
## Wrap-up — answer in this Markdown cell

1. **Country normalization.** Describe your reference-dimension approach and how you'd
   extend the dimension when a new unmapped spelling appears (tie it to your A2
   coverage — 146 rows landed in `UNKNOWN`).
2. **Inner vs left.** Why was the **inner** join right for the per-segment metrics?
   Give one report where a **left** join would be required instead.
3. **The silent trap.** In one sentence, explain what the naive `pd.to_datetime`
   one-liner did to the data and why it produced no error.

**Key takeaways**
- Normalize **both sides** to a common key before a reference-dimension join.
- Currency and dates are where "it ran fine" hides wrong answers — parse
  deliberately, and **check counts** after every parse.
- `np.select` expresses prioritized segment logic in one vectorized call — no `apply`.
- Guard every production merge with `validate=`; triage the misses with an anti-join.
